# 03 Hybrid retrieval and reranking

## Learning objectives

- compare text, vector, hybrid, and hybrid-plus-reranker configurations;
- explain Reciprocal Rank Fusion without comparing incompatible raw scores;
- keep candidate generation separate from the final context size;
- turn on Azure semantic ranking only through an explicit provider option.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()

## One fixed dataset, four retrieval configurations

Text retrieval is precise for error codes and product names. Vector retrieval
helps with symptoms and paraphrases. Azure AI Search and Databricks AI Search
both offer managed hybrid retrieval; the exact implementation and score ranges
remain provider-native. The useful question is which configuration improves
row-level outcomes on this application's cases.


In [ ]:
from agentic_ops_rag import RetrievalMode
from agentic_ops_rag.evaluation import benchmark, load_cases

pipeline = session.offline_pipeline()
cases = load_cases(course_root / "data" / "evaluation_cases.jsonl")
configurations = {
    "A_text": (RetrievalMode.TEXT, False),
    "B_vector": (RetrievalMode.VECTOR, False),
    "C_hybrid": (RetrievalMode.HYBRID, False),
    "D_hybrid_reranked": (RetrievalMode.HYBRID, True),
}
retrieval_matrix = {
    name: benchmark(pipeline, cases, mode=mode, semantic_rerank=rerank)
    for name, (mode, rerank) in configurations.items()
}
for metric in ("retrieval/recall_at_3", "retrieval/mrr"):
    print(metric)
    for name, report in retrieval_matrix.items():
        print(f"  {name:18s} {report[metric]:.4f}")
retrieval_matrix

The four configurations now genuinely disagree. Quality metrics are computed
from real row-level retrieval outcomes on this corpus. The latency column is
different in kind: it is labelled `simulated_offline_fixture` and only sketches
the shape of a cost trade-off — real latency evidence comes from measured
traces on the connected path, never from this fixture. Inspect individual
cases before choosing a winner: an average can hide an exact-code regression,
an authorization failure, or lost answerable coverage.


In [ ]:
import pandas as pd


def case_row(case):
    row = {"expected": ", ".join(case.expected_document_ids) or "(abstain)"}
    for name, (mode, rerank) in configurations.items():
        result = pipeline.invoke(
            case.question,
            tenant_id=case.tenant_id,
            region=case.region,
            allowed_groups=case.allowed_groups,
            mode=mode,
            semantic_rerank=rerank,
        )
        retrieved = list(result.retrieved_document_ids)
        expected = set(case.expected_document_ids)
        hits = len(expected.intersection(retrieved))
        marker = f"{hits}/{len(expected)}" if expected else "-"
        row[name] = f"[{marker}] " + (", ".join(retrieved) or "(abstained)")
    return row


per_case = pd.DataFrame({case.case_id: case_row(case) for case in cases}).transpose()
with pd.option_context("display.max_colwidth", 90, "display.width", 400):
    print(per_case.to_string())

divergent = per_case[per_case["A_text"] != per_case["B_vector"]]
assert per_case.loc["paraphrase-checkout-outage", "A_text"].startswith("[0/1]")
assert per_case.loc["paraphrase-checkout-outage", "B_vector"].startswith("[1/1]")
assert per_case.loc["restart-needs-approval", "B_vector"].startswith("[1/2]")
assert per_case.loc["restart-needs-approval", "A_text"].startswith("[2/2]")
for case_id in ("paraphrase-checkout-outage", "restart-needs-approval"):
    assert per_case.loc[case_id, "C_hybrid"].split("]")[0].lstrip("[") in {
        "1/1",
        "2/2",
    }
print()
print("text misses the paraphrase case; vector loses the approval policy on")
print("the exact-code case; hybrid fusion is the only configuration that")
print("recovers both:", sorted(divergent.index))

## Why fusion wins: real ranks, one real query

Take the deployment-outage question from the fixed cases (case
`semantic-checkout-outage`) and compute the two orderings the retriever
actually fuses: the lexical ranking and the embedding ranking over the
authorized corpus. No mock document IDs — this is the corpus you just
benchmarked.


In [ ]:
from agentic_ops_rag.offline import (
    cosine,
    deterministic_embedding,
    lexical_score,
    reciprocal_rank_fusion,
)

fusion_query = (
    "Checkout went down immediately after a deployment. " "How should on-call recover?"
)
scope_groups = {"ops-payments", "incident-commanders"}
scoped_documents = [
    document
    for document in pipeline.retriever.documents
    if document.active
    and document.tenant_id == "tenant-alpha"
    and document.region == "eastus"
    and scope_groups.intersection(document.allowed_groups)
]
lexical = {
    document.document_id: lexical_score(fusion_query, document)
    for document in scoped_documents
}
query_vector = deterministic_embedding(fusion_query)
semantic = {
    document.document_id: cosine(
        query_vector,
        deterministic_embedding(f"{document.title} {document.content}"),
    )
    for document in scoped_documents
}
lexical_order = sorted(lexical, key=lambda key: (-lexical[key], key))
semantic_order = sorted(semantic, key=lambda key: (-semantic[key], key))
fused = reciprocal_rank_fusion((lexical_order, semantic_order))
fused_order = sorted(fused, key=lambda key: (-fused[key], key))


def show(label, order, scores):
    print(label)
    for rank, document_id in enumerate(order[:4], start=1):
        print(f"  {rank}. {document_id:32s} {scores[document_id]: .3f}")


show("lexical ranking", lexical_order, lexical)
show("embedding ranking", semantic_order, semantic)
show("fused (RRF) ranking", fused_order, fused)

winner = fused_order[0]
assert winner == "alpha-payments-503-current"
assert lexical_order[0] != winner and semantic_order[0] != winner
print()
print(
    f"fused #1 {winner}: lexical #{lexical_order.index(winner) + 1}, "
    f"embedding #{semantic_order.index(winner) + 1} - first on neither list"
)
print(
    f"lexical #1 {lexical_order[0]} sits at embedding "
    f"#{semantic_order.index(lexical_order[0]) + 1} "
    f"and falls to fused #{fused_order.index(lexical_order[0]) + 1}"
)
print(
    f"embedding #1 {semantic_order[0]} sits at lexical "
    f"#{lexical_order.index(semantic_order[0]) + 1} "
    f"and falls to fused #{fused_order.index(semantic_order[0]) + 1}"
)

The runbook that answers the case is first on neither list — the lexical list
is topped by an incidental `on-call` identifier match and the embedding list by
the status-page mirror — yet it wins the fusion because it is strong on both.
RRF combines rank positions, not BM25 and cosine magnitudes, which is what
makes that outcome stable. Azure semantic ranking happens after hybrid fusion
and emits a separate reranker score. Never copy one absolute threshold across
BM25, vector, RRF, semantic ranker, and a different search provider.


In [ ]:
# YOUR TURN — TODO: choose candidate and context counts, then run the wide plan.
candidate_k = 50
context_k = 3
action_query = "Restart payments for ERR-PAY-503 now."
wide_plan = pipeline.invoke(
    action_query,
    tenant_id="tenant-alpha",
    region="eastus",
    allowed_groups=("ops-payments", "incident-commanders"),
    mode="hybrid",
    candidate_k=candidate_k,
    final_k=context_k,
)
print("wide plan retrieved:", list(wide_plan.retrieved_document_ids))
print(f"latency {wide_plan.latency_ms:.1f}ms " f"({wide_plan.measurement_source})")

In [ ]:
# CHECK YOUR WORK
assert candidate_k == 50, "Semantic ranker needs a broad candidate set to test"
assert 1 <= context_k <= 10, "Keep the final model context deliberately bounded"
assert "alpha-action-approval" in wide_plan.retrieved_document_ids
"Candidate generation and final context are separate decisions."

In [ ]:
# Reference solution
narrow_plan = pipeline.invoke(
    action_query,
    tenant_id="tenant-alpha",
    region="eastus",
    allowed_groups=("ops-payments", "incident-commanders"),
    mode="hybrid",
    candidate_k=2,
    final_k=2,
)
print("narrow plan retrieved:", list(narrow_plan.retrieved_document_ids))
print(f"latency {narrow_plan.latency_ms:.1f}ms ({narrow_plan.measurement_source})")
assert narrow_plan.latency_ms < wide_plan.latency_ms
assert "alpha-payments-503-current" in wide_plan.retrieved_document_ids
assert "alpha-payments-503-current" not in narrow_plan.retrieved_document_ids
print()
print("narrowing candidate_k to 2 was cheaper on fixture latency - and it")
print("silently dropped the ERR-PAY-503 runbook itself from the context.")
print("Candidate breadth is a quality decision, not only a cost knob.")

## Azure AI Search connected query

The current adapter accepts one `top_k`, so this explicit advanced call asks for
50 candidates and slices the final context in application code. That makes the
candidate/context distinction visible rather than silently pretending the SDK
has two knobs. `preFilter` protects tenant scope before vector ranking. The
stable workshop path uses classic hybrid retrieval; agentic retrieval remains
an optional platform-reviewed extension while parts of it are preview.


In [ ]:
RUN_CONNECTED = False
azure_context = None
if RUN_CONNECTED:
    from agentic_ops_rag import authorized_search

    resources = session.connected_components(allow_network=True)
    candidates = authorized_search(
        resources["retriever"],
        "Checkout went down after a deployment",
        tenant_id="tenant-alpha",
        region="eastus",
        allowed_groups=("ops-payments",),
        mode="hybrid",
        top_k=candidate_k,
        provider_options={
            "query_type": "semantic",
            "semantic_configuration_name": "operations-semantic",
            "vector_filter_mode": "preFilter",
        },
    )
    azure_context = candidates[:context_k]
azure_context

Switching to `aai-platform.databricks-search.example.yml` keeps
`operations-knowledge` unchanged. Provider-specific reranker options are an
explicit escape hatch and should be evaluated as separate changes. The
application compares outcomes and trace evidence, never raw scores across
providers. A positive provider score only orders candidates; it cannot make an
unrelated result answerable. The deterministic shell requires identifier or
query/evidence support and abstains when that support is uncertain.


## Knowledge check

Answer from the evidence you produced, not from memory:

1. Why does RRF use ranks instead of adding BM25 and vector scores?
2. Why are candidate_k and context_k separate?
3. Which filter must be applied before retrieval and why?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You ran a four-configuration ablation whose configurations genuinely disagree
at the case level, watched RRF fuse two real rankings so the runbook that tops
neither list wins, and saw candidate breadth silently decide which evidence
reaches the model. Lesson 04 turns normalized documents into MLflow traces,
deterministic gates, and optional RAG judges.
